# Notebook 05 — XGBoost + Bayesian Hyperparameter Tuning (EXP-3)
**CMPE-255 | Mental Health & Crisis Signal Mining on Social Media**  
**Owner: Gunanidhi Ramakrishnan**  
**Environment: Google Colab**

---

### What this notebook does

| Experiment | Model | Feature Input | Expected macro-F1 |
|---|---|---|---|
| **EXP-3** | XGBoost (Bayesian-tuned) | TF-IDF + NRC + Surface stats (+ LDA in Sprint 2) | 0.70 – 0.78 |

We first run a quick **baseline XGBoost** on TF-IDF only (default hyperparameters)
to establish a floor.  We then use **Optuna** (Tree-structured Parzen Estimator, TPE)
to search the hyperparameter space over ~40 trials, which is far more efficient
than a full grid search.

### Why XGBoost over Logistic Regression / SVM?
- XGBoost can natively combine **heterogeneous feature types** (sparse TF-IDF +
  dense NRC lexicon counts + post-level surface statistics) in a single model.
- Decision trees capture **non-linear feature interactions** that bag-of-words
  linear models miss.
- Tree SHAP (Notebook 09) provides fast, exact feature attributions — much faster
  than kernel-based SHAP for Logistic Regression.

---

### Prerequisites
Run **Notebook 02** (Data Preprocessing) and **Notebook 03** (Feature Engineering) first.

Required files in `DRIVE_ROOT/data/`:
- `train.parquet`, `test.parquet`

Required files in `DRIVE_ROOT/artifacts/`:
- `X_train_tfidf.npz`, `X_test_tfidf.npz`
- `X_train_engineered.npz`, `X_test_engineered.npz`
- `feature_names.json` (optional, used for feature importance labelling)

**Sprint 2 addition:** After running Notebook 07 (LDA Topic Modelling),
set `USE_LDA_FEATURES = True` and rerun this notebook to add LDA topic
probability features to the feature matrix (the EXP-3 ablation).

In [ ]:
# ============================================================
# CELL 1 — Package Installation
# ------------------------------------------------------------
# Google Colab already ships with xgboost and scikit-learn.
# We only need to install optuna (Bayesian hyperparameter search).
# ============================================================
!pip install optuna --quiet
print('Packages ready.')

In [ ]:
# ============================================================
# CELL 2 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

In [ ]:
# ============================================================
# CELL 3 — Project Path Configuration
# ============================================================
from pathlib import Path

# --- EDIT THIS LINE if your folder has a different name ---
DRIVE_ROOT = Path('/content/drive/MyDrive/CMPE255_Project')
# ----------------------------------------------------------

# Sprint 2 toggle: set to True after Notebook 07 (LDA) has been run
USE_LDA_FEATURES = True

DATA_DIR      = DRIVE_ROOT / 'data'
ARTIFACTS_DIR = DRIVE_ROOT / 'artifacts'
MODELS_DIR    = DRIVE_ROOT / 'models'
RESULTS_DIR   = DRIVE_ROOT / 'results'
FIGURES_DIR   = DRIVE_ROOT / 'figures'

for d in [MODELS_DIR, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Verify input files
required = [
    DATA_DIR / 'train.parquet',
    DATA_DIR / 'test.parquet',
    ARTIFACTS_DIR / 'X_train_tfidf.npz',
    ARTIFACTS_DIR / 'X_test_tfidf.npz',
    ARTIFACTS_DIR / 'X_train_engineered.npz',
    ARTIFACTS_DIR / 'X_test_engineered.npz',
]
optional = [ARTIFACTS_DIR / 'lda_topic_features.npz',
            ARTIFACTS_DIR / 'feature_names.json']

print('Required input files:')
for p in required:
    print(f'  {"✅" if p.exists() else "❌ MISSING"}  {p.name}')

print('\nOptional files:')
for p in optional:
    print(f'  {"✅" if p.exists() else "⚪ not yet"}  {p.name}')

if USE_LDA_FEATURES and not (ARTIFACTS_DIR / 'lda_topic_features.npz').exists():
    raise FileNotFoundError(
        'USE_LDA_FEATURES=True but lda_topic_features.npz was not found.\n'
        'Run Notebook 07 first, then re-run this notebook.'
    )

print(f'\nUSE_LDA_FEATURES = {USE_LDA_FEATURES}')

In [ ]:
# ============================================================
# CELL 4 — Standard Library Imports
# ============================================================
import random
import os
import json
from datetime import datetime

import numpy as np
import pandas as pd
import scipy.sparse
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import joblib
import warnings

# XGBoost — gradient-boosted decision trees
import xgboost as xgb

# Optuna — Bayesian hyperparameter optimisation via TPE
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress per-trial output

# scikit-learn utilities
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    accuracy_score, roc_auc_score,
    classification_report, confusion_matrix,
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print(f'XGBoost version : {xgb.__version__}')
print(f'Optuna version  : {optuna.__version__}')
print('All imports successful.')

In [ ]:
# ============================================================
# CELL 5 — Project Constants
# ============================================================

RANDOM_SEED = 42

def seed_everything(seed=RANDOM_SEED):
    """Seed Python built-ins, NumPy, and XGBoost for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything()

# 7-class Sarkar dataset label map
CLASS_NAMES = ['Anxiety', 'Bipolar', 'Depression', 'Normal',
               'Personality disorder', 'Stress', 'Suicidal']
LABEL_TO_INT = {name: i for i, name in enumerate(CLASS_NAMES)}
LABEL_MAP = {i: name for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

CLASS_COLORS = {
    'Anxiety':              '#2ECC71',
    'Bipolar':              '#9B59B6',
    'Depression':           '#3498DB',
    'Normal':               '#95A5A6',
    'Personality disorder': '#E67E22',
    'Stress':               '#E74C3C',
    'Suicidal':             '#2C3E50',
}

print(f'RANDOM_SEED = {RANDOM_SEED}')
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')

In [ ]:
# ============================================================
# CELL 6 — Metrics & Persistence Helpers  (inlined from utils/common.py)
# ============================================================

METRICS_CSV = RESULTS_DIR / 'metrics.csv'

def compute_metrics(y_true, y_pred, y_proba=None, label_names=None):
    """
    Compute the standard metric suite used across all experiments.

    Primary metric: macro-averaged F1.  It weights all seven classes equally,
    so minority-class performance is not hidden by majority-class dominance.

    Parameters
    ----------
    y_true  : 1-D integer array of true class labels (0-6)
    y_pred  : 1-D integer array of predicted class labels
    y_proba : 2-D float array of predicted probabilities, shape (n, 7)
              Required for ROC-AUC; pass None if unavailable.
    label_names : list of class name strings (defaults to CLASS_NAMES)

    Returns
    -------
    dict with macro_f1, weighted_f1, accuracy, roc_auc_macro,
         per_class_precision / recall / f1, confusion_matrix, classification_report
    """
    if label_names is None:
        label_names = CLASS_NAMES
    n = len(label_names)

    macro_f1    = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    acc         = accuracy_score(y_true, y_pred)

    per_class_p = precision_score(y_true, y_pred, average=None, zero_division=0).tolist()
    per_class_r = recall_score   (y_true, y_pred, average=None, zero_division=0).tolist()
    per_class_f = f1_score       (y_true, y_pred, average=None, zero_division=0).tolist()

    roc_auc = None
    if y_proba is not None:
        try:
            y_bin   = label_binarize(y_true, classes=list(range(n)))
            roc_auc = roc_auc_score(y_bin, y_proba, average='macro', multi_class='ovr')
        except Exception:
            roc_auc = None

    cm         = confusion_matrix(y_true, y_pred, labels=list(range(n)))
    report_str = classification_report(y_true, y_pred, target_names=label_names, zero_division=0)

    return {
        'macro_f1':              round(macro_f1,    4),
        'weighted_f1':           round(weighted_f1, 4),
        'accuracy':              round(acc,         4),
        'roc_auc_macro':         round(roc_auc, 4) if roc_auc is not None else None,
        'per_class_precision':   per_class_p,
        'per_class_recall':      per_class_r,
        'per_class_f1':          per_class_f,
        'confusion_matrix':      cm.tolist(),
        'classification_report': report_str,
    }


def save_metrics(exp_name, model, condition, metrics_dict,
                 split='in_dist_test', best_cv_score=None, notes=''):
    """
    Append one row to results/metrics.csv.
    All experiments share the same CSV so Notebook 08 can load and compare them.
    """
    row = {
        'exp_name':      exp_name,
        'model':         model,
        'condition':     condition,
        'split':         split,
        'macro_f1':      metrics_dict.get('macro_f1'),
        'weighted_f1':   metrics_dict.get('weighted_f1'),
        'roc_auc_macro': metrics_dict.get('roc_auc_macro'),
        'accuracy':      metrics_dict.get('accuracy'),
        'best_cv_score': best_cv_score,
        'run_timestamp': datetime.utcnow().isoformat(timespec='seconds'),
        'notes':         notes,
    }
    row_df = pd.DataFrame([row])

    if METRICS_CSV.exists():
        existing = pd.read_csv(METRICS_CSV)
        updated  = pd.concat([existing, row_df], ignore_index=True)
    else:
        updated = row_df

    updated.to_csv(METRICS_CSV, index=False)
    print(f'  [saved] {exp_name} | {model} | {condition} | macro_f1={row["macro_f1"]:.4f}')

print('compute_metrics() and save_metrics() ready.')

In [ ]:
# ============================================================
# CELL 7 — Plotting Helpers  (inlined from utils/common.py)
# ============================================================

def plot_confusion_matrix(cm, labels, title='Confusion Matrix',
                          save_path=None, figsize=(8, 6)):
    """Plot a row-normalised confusion matrix heatmap."""
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Oranges',
                xticklabels=labels, yticklabels=labels,
                ax=ax, vmin=0, vmax=1, linewidths=0.5)
    ax.set_title(title, fontsize=13, pad=12)
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label',      fontsize=11)
    plt.tight_layout()
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'  [saved] {save_path}')
    plt.show()
    plt.close(fig)


def plot_roc_curves(y_true, y_proba, labels,
                    title='ROC Curves (one-vs-rest)',
                    save_path=None, figsize=(8, 6)):
    """Plot one-vs-rest ROC curves for each of the seven classes."""
    from sklearn.metrics import roc_curve, auc
    n_classes  = len(labels)
    y_bin      = label_binarize(y_true, classes=list(range(n_classes)))
    color_vals = list(CLASS_COLORS.values())
    fig, ax    = plt.subplots(figsize=figsize)
    for i, name in enumerate(labels):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
        ax.plot(fpr, tpr, lw=1.8, color=color_vals[i % len(color_vals)],
                label=f'{name}  (AUC = {auc(fpr, tpr):.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random baseline')
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate',  fontsize=11)
    ax.set_title(title, fontsize=13)
    ax.legend(loc='lower right', fontsize=9)
    plt.tight_layout()
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'  [saved] {save_path}')
    plt.show()
    plt.close(fig)

print('Plotting helpers ready.')

## 1. Load Data and Build Combined Feature Matrix

XGBoost accepts **dense** feature matrices as numpy arrays or **DMatrix** objects.
We need to:
1. Load the sparse TF-IDF matrix (from NB-03)
2. Load the dense engineered features (NRC + surface stats)
3. Horizontally stack them: `[TF-IDF | NRC | surface | (LDA)]`
4. Convert to dense (`.toarray()`) or keep sparse — XGBoost handles both.

> **Memory note:** A dense `(4766 × 50000)` TF-IDF matrix takes ~1.8 GB.
> For Colab free tier, convert to dense only when needed and avoid keeping
> multiple copies in memory simultaneously.

In [ ]:
# ============================================================
# CELL 8 — Load Labels, Feature Matrices, and Build Combined Matrix
# ============================================================

# ---- Labels ----
train_df = pd.read_parquet(DATA_DIR / 'train.parquet')
test_df  = pd.read_parquet(DATA_DIR / 'test.parquet')

# Encode string labels to integers
y_train = train_df['label'].map(LABEL_TO_INT).values
y_test  = test_df['label'].map(LABEL_TO_INT).values

print(f'Training samples : {len(y_train)}')
print(f'Test samples     : {len(y_test)}')

# ---- TF-IDF sparse matrices ----
X_train_tfidf = scipy.sparse.load_npz(ARTIFACTS_DIR / 'X_train_tfidf.npz').tocsr()
X_test_tfidf  = scipy.sparse.load_npz(ARTIFACTS_DIR / 'X_test_tfidf.npz').tocsr()

# ---- Engineered dense features (NRC + surface stats) ----
X_train_eng = np.load(ARTIFACTS_DIR / 'X_train_engineered.npz')['arr_0']
X_test_eng  = np.load(ARTIFACTS_DIR / 'X_test_engineered.npz')['arr_0']

print(f'TF-IDF features  : {X_train_tfidf.shape[1]}')
print(f'Engineered feats : {X_train_eng.shape[1]}')

# ---- Optional: LDA topic probability features ----
if USE_LDA_FEATURES:
    lda_data        = np.load(ARTIFACTS_DIR / 'lda_topic_features.npz')
    X_train_lda     = lda_data['X_train']
    X_test_lda      = lda_data['X_test']
    lda_feature_dim = X_train_lda.shape[1]
    print(f'LDA topic feats  : {lda_feature_dim}')
else:
    X_train_lda = X_test_lda = None
    lda_feature_dim = 0
    print('LDA features     : not used (set USE_LDA_FEATURES=True after NB-07)')

# ---- Combine into final feature matrix ----
parts_train = [X_train_tfidf, scipy.sparse.csr_matrix(X_train_eng)]
parts_test  = [X_test_tfidf,  scipy.sparse.csr_matrix(X_test_eng)]

if X_train_lda is not None:
    parts_train.append(scipy.sparse.csr_matrix(X_train_lda))
    parts_test.append(scipy.sparse.csr_matrix(X_test_lda))

X_train_combined = scipy.sparse.hstack(parts_train, format='csr')
X_test_combined  = scipy.sparse.hstack(parts_test,  format='csr')

total_features = X_train_tfidf.shape[1] + X_train_eng.shape[1] + lda_feature_dim
print(f'\nCombined feature matrix: {X_train_combined.shape}  ({total_features} total features)')

# ---- Feature names ----
feat_names_path = ARTIFACTS_DIR / 'feature_names.json'
if feat_names_path.exists():
    with open(feat_names_path) as f:
        feature_names = json.load(f)
    print(f'Loaded {len(feature_names)} feature names from feature_names.json')
else:
    feature_names = (
        [f'tfidf_{i}' for i in range(X_train_tfidf.shape[1])] +
        [f'eng_{i}'   for i in range(X_train_eng.shape[1])]   +
        ([f'lda_{i}'  for i in range(lda_feature_dim)] if USE_LDA_FEATURES else [])
    )
    print(f'feature_names.json not found — using generic names')

## 2. Baseline XGBoost (Default Hyperparameters, TF-IDF Only)

Before running the expensive Bayesian search, we first establish a **floor**:
XGBoost with default hyperparameters on TF-IDF features only (no NRC/surface/LDA).

This baseline:
- Quantifies how much Bayesian tuning actually helps.
- Tells us whether adding engineered features improves over TF-IDF alone.

In [ ]:
# ============================================================
# CELL 9 — Baseline XGBoost (TF-IDF Only, Default Hyperparameters)
# ============================================================

print('Training baseline XGBoost (default hyperparameters, TF-IDF only)...')

xgb_baseline = xgb.XGBClassifier(
    n_estimators=100,
    objective='multi:softprob',
    num_class=NUM_CLASSES,
    eval_metric='mlogloss',
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbosity=0,
)

xgb_baseline.fit(X_train_tfidf, y_train)

y_pred_test  = xgb_baseline.predict(X_test_tfidf)
y_proba_test = xgb_baseline.predict_proba(X_test_tfidf)

metrics_base = compute_metrics(y_test, y_pred_test, y_proba_test)

print(f'\nBaseline XGBoost (TF-IDF only, default params):')
print(f'  Test macro-F1  = {metrics_base["macro_f1"]:.4f}')
print(f'  Test accuracy  = {metrics_base["accuracy"]:.4f}')
print(f'  ROC-AUC (macro) = {metrics_base["roc_auc_macro"]:.4f}')
print()
print(metrics_base['classification_report'])

save_metrics('EXP-3', 'XGBoost', 'tfidf_only_default', metrics_base,
             notes='baseline_default_hyperparameters')

BASELINE_F1 = metrics_base['macro_f1']
print(f'Baseline macro-F1 stored: {BASELINE_F1:.4f}')

## 3. Bayesian Hyperparameter Search with Optuna (EXP-3)

### Why Bayesian search instead of grid search?

A full grid search over 6 hyperparameters with 5 folds and reasonable ranges
would require **thousands of XGBoost fits** and would not finish in a Colab session.

Optuna uses the **Tree-structured Parzen Estimator (TPE)** algorithm:
- After each trial, it builds a probabilistic model of which hyperparameter
  regions produced good results.
- The next trial is sampled from regions likely to improve performance.
- This means 40 trials explore the space much more efficiently than a 4×4×4
  grid would.

### Hyperparameters being tuned

| Parameter | Search Range | What it controls |
|---|---|---|
| `max_depth` | 3 – 10 | Maximum depth of each tree. Larger = more complex, prone to overfitting. |
| `learning_rate` | 0.01 – 0.30 (log scale) | Shrinkage applied to each tree. Smaller = slower but often better. |
| `n_estimators` | 100 – 600 | Number of boosting rounds (trees). More is better up to the point of overfitting. |
| `subsample` | 0.6 – 1.0 | Fraction of training samples used per tree. < 1.0 adds stochasticity to reduce overfitting. |
| `colsample_bytree` | 0.5 – 1.0 | Fraction of features used per tree. Helps with high-dimensional TF-IDF. |
| `min_child_weight` | 1 – 10 | Minimum sum of sample weights in a leaf. Higher = more conservative splits. |

In [ ]:
# ============================================================
# CELL 10 — Define the Optuna Objective Function
# ============================================================

SKF = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

def optuna_objective(trial):
    """Optuna objective: 3-fold CV macro-F1 on combined features."""
    params = {
        'max_depth':        trial.suggest_int('max_depth',   4, 9),
        'learning_rate':    trial.suggest_float('learning_rate',  0.01, 0.30, log=True),
        'n_estimators':     trial.suggest_int('n_estimators', 100, 400),
        'subsample':        trial.suggest_float('subsample',        0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.9),
        'min_child_weight': trial.suggest_int('min_child_weight',   1, 10),
        'objective':         'multi:softprob',
        'num_class':          NUM_CLASSES,
        'eval_metric':        'mlogloss',
        'random_state':       RANDOM_SEED,
        'n_jobs':             -1,
        'verbosity':          0,
    }

    model = xgb.XGBClassifier(**params)
    cv_scores = cross_val_score(
        model, X_train_combined, y_train,
        cv=SKF, scoring='f1_macro', n_jobs=1,
    )
    return cv_scores.mean()

print('Optuna objective function defined.')
print(f'Each trial runs 3-fold CV on the combined feature matrix ({NUM_CLASSES} classes).')

In [ ]:
# ============================================================
# CELL 11 — Run the Bayesian Search
# ------------------------------------------------------------
# n_trials=40 : run 40 trial configurations
# timeout=600  : hard stop after 10 minutes (important for Colab free tier)
#
# On a Colab T4 GPU (or CPU), 40 trials with 3-fold CV typically takes
# 15-30 minutes for a 50K-feature sparse matrix.  If it is too slow,
# reduce N_TRIALS to 20 or set USE_LDA_FEATURES=False first.
# ============================================================

N_TRIALS = 40   # reduce to 20 if Colab is slow; increase to 60 for better tuning
TIMEOUT  = 900  # seconds (15 min hard cap)

print(f'Starting Optuna study  ({N_TRIALS} trials, {TIMEOUT}s timeout)...')
print(f'Feature matrix: {X_train_combined.shape[0]} samples × {X_train_combined.shape[1]} features')
print('This may take 15–30 minutes on a free Colab CPU/T4 instance.\n')

# Create an Optuna study that maximises the objective (macro-F1)
# TPESampler with our fixed RANDOM_SEED makes the study reproducible
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0),
)

# Run the search — Optuna calls optuna_objective(trial) N_TRIALS times
study.optimize(
    optuna_objective,
    n_trials=N_TRIALS,
    timeout=TIMEOUT,
    show_progress_bar=True,  # shows a tqdm progress bar in Colab
)

print(f'\nOptuna search complete!')
print(f'  Trials completed   : {len(study.trials)}')
print(f'  Best CV macro-F1   : {study.best_value:.4f}')
print(f'  Best hyperparameters:')
for k, v in study.best_params.items():
    print(f'    {k:<22} = {v}')

In [ ]:
# ============================================================
# CELL 12 — Optuna Diagnostics: Optimization History
# ============================================================

# Extract trial numbers and their objective values
trial_numbers = [t.number for t in study.trials if t.value is not None]
trial_values  = [t.value  for t in study.trials if t.value is not None]

# Running best (i.e., best seen up to each trial)
running_best = [max(trial_values[:i+1]) for i in range(len(trial_values))]

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(trial_numbers, trial_values, alpha=0.5, s=20, color='steelblue',
           label='Individual trial macro-F1')
ax.plot(trial_numbers, running_best, color='darkorange', lw=2.0,
        label='Running best')
ax.axhline(y=BASELINE_F1, color='red', linestyle='--', lw=1.5,
           label=f'Default XGBoost baseline ({BASELINE_F1:.3f})')
ax.set_xlabel('Trial number', fontsize=11)
ax.set_ylabel('3-fold CV macro-F1', fontsize=11)
ax.set_title('Optuna Optimisation History (XGBoost EXP-3)', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()

opt_history_path = FIGURES_DIR / 'xgb_optuna_history.png'
fig.savefig(opt_history_path, dpi=150, bbox_inches='tight')
print(f'Saved → {opt_history_path.name}')
plt.show()

# ---- Parallel coordinate plot (shows which param regions performed best) ----
# This visualisation is useful for understanding which hyperparameter values
# most consistently appear in the top trials.
try:
    import optuna.visualization.matplotlib as optuna_viz_mpl
    fig2 = optuna_viz_mpl.plot_parallel_coordinate(study)
    fig2.set_size_inches(12, 5)
    fig2.savefig(FIGURES_DIR / 'xgb_optuna_parallel_coord.png', dpi=120, bbox_inches='tight')
    print('Parallel coordinate plot saved.')
    plt.show()
except Exception as e:
    print(f'(Parallel coordinate plot skipped: {e})')

In [ ]:
# ============================================================
# CELL 13 — Retrain Best Model on Full Training Set
# ============================================================

print('Retraining XGBoost with best hyperparameters on the full training set...')

best_params = study.best_params.copy()
best_params.update({
    'objective':         'multi:softprob',
    'num_class':          NUM_CLASSES,
    'eval_metric':        'mlogloss',
    'random_state':       RANDOM_SEED,
    'n_jobs':             -1,
    'verbosity':          0,
})

best_xgb = xgb.XGBClassifier(**best_params)
best_xgb.fit(X_train_combined, y_train)

y_pred_best  = best_xgb.predict(X_test_combined)
y_proba_best = best_xgb.predict_proba(X_test_combined)

metrics_best = compute_metrics(y_test, y_pred_best, y_proba_best)

print(f'\nBest XGBoost (EXP-3) test results:')
print(f'  Macro-F1   = {metrics_best["macro_f1"]:.4f}  '
      f'(vs. baseline {BASELINE_F1:.4f}, gain = {metrics_best["macro_f1"]-BASELINE_F1:+.4f})')
print(f'  Accuracy   = {metrics_best["accuracy"]:.4f}')
print(f'  ROC-AUC    = {metrics_best["roc_auc_macro"]:.4f}')
print()
print(metrics_best['classification_report'])

if USE_LDA_FEATURES:
    exp3_condition = 'tfidf+engineered+lda'
else:
    exp3_condition = 'tfidf+engineered'

save_metrics(
    'EXP-3', 'XGBoost', exp3_condition, metrics_best,
    best_cv_score=study.best_value,
    notes=f'optuna_trials={len(study.trials)}; best_params={str(study.best_params)}',
)

In [ ]:
# ============================================================
# CELL 14 — Save Best Model and Persist Study Metadata
# ============================================================

# ---- Save the trained XGBoost model ----
# joblib is the recommended serialisation format for scikit-learn-compatible models.
# It handles numpy arrays and sparse matrices efficiently.
xgb_model_path = MODELS_DIR / 'xgb_best.joblib'
joblib.dump(best_xgb, xgb_model_path)
print(f'Best XGBoost model saved → {xgb_model_path}')

# ---- Save best hyperparameters as JSON ----
# This makes it easy to inspect the tuned values without loading the full model.
best_params_serialisable = {
    k: (int(v) if isinstance(v, (np.integer,)) else
        float(v) if isinstance(v, (np.floating,)) else v)
    for k, v in study.best_params.items()
}
best_params_path = ARTIFACTS_DIR / 'xgb_best_params.json'
with open(best_params_path, 'w') as f:
    json.dump({
        'best_params':      best_params_serialisable,
        'best_cv_macro_f1': float(study.best_value),
        'n_trials':         len(study.trials),
        'feature_condition': exp3_condition,
        'timestamp':        datetime.utcnow().isoformat(timespec='seconds'),
    }, f, indent=2)
print(f'Best params saved → {best_params_path}')

## 4. Feature Importance Analysis

XGBoost provides two types of feature importance:

| Type | What it measures |
|---|---|
| `weight` (frequency) | Number of times a feature is used across all trees. Common for TF-IDF tokens. |
| `gain` (information gain) | Average reduction in loss when a feature is used for a split. Better indicator of true importance. |

We use **`gain`** here because it directly measures how much each feature
improved the model's predictive power, not just how often it appeared.

In [ ]:
# ============================================================
# CELL 15 — Feature Importance: Top 30 Features by Gain
# ============================================================

# Get gain-based feature importance from the trained XGBoost model
# Returns a dict: {feature_index_or_name: importance_score}
importance_dict = best_xgb.get_booster().get_score(importance_type='gain')

# XGBoost names features 'f0', 'f1', ... when trained on sparse matrices
# Map these back to human-readable feature names where possible
importance_df = pd.DataFrame(
    [(int(k[1:]), v) for k, v in importance_dict.items()],
    columns=['feature_idx', 'importance_gain']
).sort_values('importance_gain', ascending=False)

# Add human-readable names
importance_df['feature_name'] = importance_df['feature_idx'].apply(
    lambda i: feature_names[i] if i < len(feature_names) else f'f{i}'
)

# Identify which feature type each one is (TF-IDF, NRC, surface, LDA)
n_tfidf = X_train_tfidf.shape[1]
n_eng   = X_train_eng.shape[1]

def get_feature_type(idx):
    if idx < n_tfidf:
        return 'TF-IDF token'
    elif idx < n_tfidf + n_eng:
        return 'Engineered (NRC/surface)'
    else:
        return 'LDA topic'

importance_df['feature_type'] = importance_df['feature_idx'].apply(get_feature_type)

# Save full importance list to CSV (will be used by NB-09 SHAP analysis)
feat_importance_path = ARTIFACTS_DIR / 'xgb_feature_importance.csv'
importance_df.to_csv(feat_importance_path, index=False)
print(f'Feature importance saved → {feat_importance_path.name}')
print(f'Total features with non-zero importance: {len(importance_df)}')

# ---- Plot top 30 features ----
top30 = importance_df.head(30).iloc[::-1]  # reverse for horizontal bar chart

# Colour bars by feature type
type_colors = {
    'TF-IDF token':            '#3498DB',
    'Engineered (NRC/surface)': '#E67E22',
    'LDA topic':                '#2ECC71',
}
bar_colors = [type_colors.get(t, '#888') for t in top30['feature_type']]

fig, ax = plt.subplots(figsize=(10, 9))
bars = ax.barh(top30['feature_name'], top30['importance_gain'],
               color=bar_colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('XGBoost Gain Importance', fontsize=11)
ax.set_title('Top 30 Features — XGBoost EXP-3 (by gain)', fontsize=12)

# Add a legend for feature types
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=c, label=t) for t, c in type_colors.items()
                  if t in top30['feature_type'].values]
ax.legend(handles=legend_handles, loc='lower right', fontsize=9)

plt.tight_layout()
feat_imp_plot_path = FIGURES_DIR / 'xgb_feature_importance_top30.png'
fig.savefig(feat_imp_plot_path, dpi=150, bbox_inches='tight')
print(f'Plot saved → {feat_imp_plot_path.name}')
plt.show()

# Print a text preview of top-10
print('\nTop 10 features by gain:')
print(importance_df[['feature_name', 'feature_type', 'importance_gain']].head(10).to_string(index=False))

In [ ]:
# ============================================================
# CELL 16 — Confusion Matrix and ROC Curves for Best XGBoost
# ============================================================

# Confusion matrix
cm_xgb = np.array(metrics_best['confusion_matrix'])
plot_confusion_matrix(
    cm_xgb, CLASS_NAMES,
    title=f'EXP-3 XGBoost ({exp3_condition})\nmacro-F1 = {metrics_best["macro_f1"]:.4f}',
    save_path=FIGURES_DIR / 'cm_xgb_best.png',
)

# ROC curves
plot_roc_curves(
    y_test, y_proba_best, CLASS_NAMES,
    title=f'EXP-3 XGBoost ROC Curves ({exp3_condition})',
    save_path=FIGURES_DIR / 'roc_xgb_best.png',
)

# Per-class breakdown table
print('\nPer-class metrics for best XGBoost (EXP-3):')
per_class_df = pd.DataFrame({
    'Class':     CLASS_NAMES,
    'Precision': [round(p, 4) for p in metrics_best['per_class_precision']],
    'Recall':    [round(r, 4) for r in metrics_best['per_class_recall']],
    'F1':        [round(f, 4) for f in metrics_best['per_class_f1']],
})
print(per_class_df.to_string(index=False))

## 5. Sprint 2: LDA Feature Ablation Placeholder

After **Notebook 07 (LDA Topic Modeling)** is complete:

1. Go back to **Cell 3** of this notebook and set `USE_LDA_FEATURES = True`.
2. Re-run the notebook from **Cell 3 onwards**.
3. The combined feature matrix will include LDA topic probability vectors,
   and a new EXP-3 row will be written to `metrics.csv` with
   `condition = 'tfidf+engineered+lda'`.
4. Notebook 08 (Model Comparison) will automatically show both the
   `tfidf+engineered` and `tfidf+engineered+lda` rows side by side
   as the EXP-3 ablation.

The expected gain from adding LDA features is modest (1-3 macro-F1 points)
but represents interpretable topical structure — making it valuable for
the paper even if the numerical gain is small.

In [ ]:
# ============================================================
# CELL 17 — LDA Ablation: Status Check
# ============================================================

lda_path = ARTIFACTS_DIR / 'lda_topic_features.npz'

if lda_path.exists() and not USE_LDA_FEATURES:
    print('ℹ️  lda_topic_features.npz exists but USE_LDA_FEATURES = False.')
    print('   Set USE_LDA_FEATURES = True in Cell 3 and re-run the notebook')
    print('   to get the EXP-3 ablation result with LDA topic features added.')
elif not lda_path.exists():
    print('⏳ lda_topic_features.npz does not exist yet.')
    print('   Run Notebook 07 (LDA Topic Modeling) first.')
    print('   Then come back, set USE_LDA_FEATURES = True, and re-run.')
else:
    print('✅ LDA features used in this run.')
    print(f'   Feature condition: {exp3_condition}')

## 6. Final Summary

In [ ]:
# ============================================================
# CELL 18 — EXP-3 Summary: Baseline vs. Tuned XGBoost
# ============================================================

# Read the metrics.csv rows relevant to this notebook
metrics_df = pd.read_csv(METRICS_CSV)
exp3_rows  = metrics_df[metrics_df['exp_name'] == 'EXP-3'].copy()

print('EXP-3 XGBoost Results Summary')
print('=' * 70)
print(exp3_rows[['condition', 'macro_f1', 'weighted_f1', 'roc_auc_macro',
                  'accuracy', 'best_cv_score']].to_string(index=False))

# Improvement over baseline
if len(exp3_rows) >= 2:
    base_f1 = exp3_rows[exp3_rows['condition'].str.contains('default')]['macro_f1'].values[0]
    best_f1 = exp3_rows[~exp3_rows['condition'].str.contains('default')]['macro_f1'].max()
    print(f'\nBayesian tuning improvement: {best_f1 - base_f1:+.4f} macro-F1 points')

print(f'\nBest params: {study.best_params}')
print(f'Best CV macro-F1 : {study.best_value:.4f}')
print(f'Test macro-F1    : {metrics_best["macro_f1"]:.4f}')

print('\n✅ Notebook 05 complete.')
print(f'   Model saved     : {MODELS_DIR}/xgb_best.joblib')
print(f'   Params saved    : {ARTIFACTS_DIR}/xgb_best_params.json')
print(f'   Metrics written : {METRICS_CSV}')
print(f'   Figures saved   : {FIGURES_DIR}')
print('\nNext: Notebook 06 (DistilBERT Fine-Tuning) or')
print('       Notebook 07 (LDA Topic Modeling) → then re-run NB-05 with USE_LDA_FEATURES=True')